### Step 1 — Configure Python path

Purpose: Add the `src/` directory to `sys.path` so `ibnr_utils` is importable without package installation, regardless of whether the notebook is run from `notebooks/` or the project root.  
Uses: `pathlib.Path`, `sys.path`.  
Produces: `src_path` resolved and appended to `sys.path`.

In [1]:
import sys
from pathlib import Path

src_path = Path.cwd().parent / "src" if Path.cwd().name == "notebooks" else Path.cwd() / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

### Step 2 — Import the workflow function

Purpose: Import the single high-level entry point used in this notebook.  
Uses: `create_triangles_from_project_database` from `ibnr_utils`.  
Produces: Function symbol available in the session.

In [2]:
from ibnr_utils import create_triangles_from_project_database

### Step 3 — Run the workflow for a single concept

Purpose: Demonstrate the minimal workflow call: auto-locate the project root, read the database, build the triangle collection, and validate — all in one function.  
Uses: `create_triangles_from_project_database(basis="month", concepts="Loss Incurred")`.  
Produces: `result` — a `TriangleWorkflowResult` with attributes `triangles`, `validation_passed`, `validation_error`, and `database`.  

Interpretation: Passing a single string for `concepts` is a convenience shorthand; the function normalises it to a set internally.

In [3]:
result = create_triangles_from_project_database(
    basis="month",
    concepts="Loss Incurred",
)

result.validation_passed, result.validation_error, list(result.triangles)

All validations passed.


(True, None, ['Loss Incurred'])

### Step 4 — Inspect the Loss Incurred triangle from the result

Purpose: Access the triangle for a specific concept from the `TriangleWorkflowResult` and display it to verify expected dimensions and `NaN` pattern.  
Produces: Loss Incurred triangle DataFrame extracted from `result.triangles`.

In [4]:
result.triangles["Loss Incurred"]

,dev_0,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,...,dev_110,dev_111,dev_112,dev_113,dev_114,dev_115,dev_116,dev_117,dev_118,dev_119
accident_period,,,,,,,,,,,,,,,,,,,,,
2016-01,153700.711642,168265.054672,205478.444379,184336.168001,201248.543480,196453.059025,176881.337833,200214.346199,187578.698914,190075.991440,...,177018.978866,171085.713030,173585.908595,169160.747063,173081.527124,176445.361233,172720.085632,173112.466318,174926.788130,175796.196292
2016-02,174526.085249,179397.620865,205041.059805,195258.548978,187048.160649,201436.435289,199293.923502,205210.386921,242546.688042,200341.751962,...,183520.470365,177750.670575,178954.148728,181846.011293,186153.778295,183854.533150,183737.684382,180857.272493,183746.248505,NaN
2016-03,139977.733603,194087.931093,215736.346315,175783.066907,217515.571324,183537.102418,202607.299157,199878.253026,206739.049289,236638.405603,...,185889.013660,185870.257200,187689.323174,188042.155009,189089.878262,185853.845022,186063.868425,184100.501488,NaN,NaN
2016-04,179441.145220,171328.983083,197378.848951,208702.402696,208493.696364,223233.698165,187690.613593,253992.426802,225375.117705,243903.465956,...,200265.915955,199299.153969,197234.175592,195849.823081,193656.903165,201506.058677,196951.796202,NaN,NaN,NaN
2016-05,185553.143033,216977.705207,209628.471276,214214.723923,232801.660391,210101.993900,201936.310336,223423.445555,230653.011303,211583.631794,...,209927.848694,207676.902404,209507.332802,209279.782933,205822.853384,201165.375549,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08,362141.012374,432661.332612,445636.521558,541592.036871,378209.883552,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09,367005.573515,415446.207539,433412.965713,518520.051557,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-10,345630.285842,346963.701868,393892.055949,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Step 5 — Run the workflow for multiple concepts with scoped validation

Purpose: Show how to request multiple concepts in a single call and how to restrict validation to the latest diagonal only.  
Uses: `create_triangles_from_project_database(basis="month", concepts=[...], validation_scope="latest_diagonal")`.  
Produces: `selected_result` — a `TriangleWorkflowResult` containing triangles for Loss Incurred, Loss Paid, Claims Reported, and Claims Paid.  

Interpretation: `validation_scope="latest_diagonal"` is the default; pass `"every_cell"` to validate every historical cell instead of only the most recent observed diagonal.

In [5]:
selected_result = create_triangles_from_project_database(
    basis="month",
    concepts=["Loss Incurred", "Loss Paid", "Claims Reported", "Claims Paid"],
    validation_scope="latest_diagonal",
)

selected_result.validation_passed, list(selected_result.triangles)

All validations passed.


(True, ['Claims Paid', 'Claims Reported', 'Loss Incurred', 'Loss Paid'])

### Step 6 — Inspect the Loss Paid triangle from the result

Purpose: Access and display a second concept's triangle to confirm that all requested concepts were built correctly.  
Produces: Loss Paid triangle DataFrame extracted from `selected_result.triangles`.

In [6]:
selected_result.triangles["Loss Paid"]

,dev_0,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,...,dev_110,dev_111,dev_112,dev_113,dev_114,dev_115,dev_116,dev_117,dev_118,dev_119
accident_period,,,,,,,,,,,,,,,,,,,,,
2016-01,51742.152671,63054.863827,87655.085175,87655.085175,94236.623267,94963.851570,94963.851570,103675.819284,103675.819284,103675.819284,...,175796.196292,175796.196292,175796.196292,175796.196292,175796.196292,175796.196292,175796.196292,175796.196292,175796.196292,175796.196292
2016-02,59638.609302,66818.515118,85081.554234,85081.554234,85081.554234,95589.625784,97523.805695,104330.468551,132902.325738,132902.325738,...,183746.248505,183746.248505,183746.248505,183746.248505,183746.248505,183746.248505,183746.248505,183746.248505,183746.248505,NaN
2016-03,44147.589409,73699.959110,90416.498511,90416.498511,101000.659082,101000.659082,101000.659082,101000.659082,107352.301891,130635.184022,...,184100.501488,184100.501488,184100.501488,184100.501488,184100.501488,184100.501488,184100.501488,184100.501488,NaN,NaN
2016-04,60089.186639,61044.929978,78591.535009,89351.901044,93469.586201,106378.384938,106378.384938,134045.619188,134045.619188,134045.619188,...,196951.796202,196951.796202,196951.796202,196951.796202,196951.796202,196951.796202,196951.796202,NaN,NaN,NaN
2016-05,60553.430274,80545.783754,82060.490940,89175.816387,104230.287268,104230.287268,104230.287268,109465.904319,117017.886935,117017.886935,...,201165.375549,201165.375549,201165.375549,201165.375549,201165.375549,201165.375549,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08,103682.071373,141946.409216,157449.330886,215119.311559,215119.311559,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09,151638.472130,193197.723659,217966.056832,291672.134735,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-10,113201.350094,122946.448753,155233.315010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
